In [2]:
import pandas as pd
red = pd.read_csv("delhimetrorail (1).csv")

In [12]:
red.tail(10)

,item_name,description,item_quantity,station_name,receiving_date,receiving_time
13703,mayur jug,mayur jug,1,KASHMERE GATE,11/01/2022,19:20:00
13704,bag with charger speaker mobile bank card aadh...,bag with charger speaker mobile bank card aadh...,1,KASHMERE GATE,11/01/2022,00:10:00
13705,purse with aahar copy photo HDFC card cash of ...,purse with aahar copy photo HDFC card cash of ...,1,NEHRU ENCLAVE,11/01/2022,17:30:00
13706,bag with cheque book case roll shoes charger c...,bag with cheque book case roll shoes charger c...,1,TERMINAL 1-IGI AIRPORT,11/01/2022,21:15:00
13707,empty bag,empty bag,1,TERMINAL 1-IGI AIRPORT,11/01/2022,21:15:00
13708,frocks,frocks,1,JLN STADIUM,11/01/2022,19:50:00
13709,jeans towel,jeans towel,1,SAROJINI NAGAR,11/01/2022,20:40:00
13710,blanket,blanket,1,KAUSHAMBI,11/01/2022,17:30:00
13711,bag with bedsheet cap jacket,bag with bedsheet cap jacket,1,SAMAYPUR BADLI,11/01/2022,17:10:00
13712,"BAG HAVING CASH, LUNCH, BOTTLE, BANK DOCUMENTS...","BAG HAVING CASH, LUNCH, BOTTLE, BANK DOCUMENTS...",1,RAJA NAHAR SINGH (BALLABHGARH),15/12/2021,20:05:00


In [13]:
red.head(10)

,item_name,description,item_quantity,station_name,receiving_date,receiving_time
0,CHAIN,METAL CHAIN,1,VAISHALI,21/03/2024,11:35:00
1,EARBUDS CASE,BLACK EARBUDS,1,BRIG. HOSHIAR SINGH,20/03/2024,12:15:00
2,CARRY BAG,OLD LADIES CLOTHES,1,RITHALA,20/03/2024,10:00:00
3,BAG,"UTENSILS , PEN , HANGER & LUNCH BOX",1,KAROL BAGH,20/03/2024,09:30:00
4,JACKET,GREY JACKET,2,YASHOBHOOMI DWARKA SECTOR - 25,20/03/2024,11:00:00
5,BAG,"WITH DENIM PANT & JACKET ,TIFFIN ,TOWEL ,LOWER...",2,YASHOBHOOMI DWARKA SECTOR - 25,20/03/2024,11:30:00
6,JUTE BAG,WITH LUNCH BOX,1,SAMAYPUR BADLI,20/03/2024,10:20:00
7,BROWN PURSE,"BANK CARDS , AADHAR CARD , PAN CARD OF MR. NAR...",1,KASHMERE GATE,20/03/2024,17:30:00
8,JUTE BAG,"WITH BOTTLE ,GLASS BOXES",1,IFFCO CHOWK,20/03/2024,20:00:00
9,WATCH,LADY WATCH,1,DELHI CANTT.,20/03/2024,10:00:00


In [14]:
red = red.drop(columns=["station_name"])

In [15]:
red.tail(10)

,item_name,description,item_quantity,receiving_date,receiving_time
13703,mayur jug,mayur jug,1,11/01/2022,19:20:00
13704,bag with charger speaker mobile bank card aadh...,bag with charger speaker mobile bank card aadh...,1,11/01/2022,00:10:00
13705,purse with aahar copy photo HDFC card cash of ...,purse with aahar copy photo HDFC card cash of ...,1,11/01/2022,17:30:00
13706,bag with cheque book case roll shoes charger c...,bag with cheque book case roll shoes charger c...,1,11/01/2022,21:15:00
13707,empty bag,empty bag,1,11/01/2022,21:15:00
13708,frocks,frocks,1,11/01/2022,19:50:00
13709,jeans towel,jeans towel,1,11/01/2022,20:40:00
13710,blanket,blanket,1,11/01/2022,17:30:00
13711,bag with bedsheet cap jacket,bag with bedsheet cap jacket,1,11/01/2022,17:10:00
13712,"BAG HAVING CASH, LUNCH, BOTTLE, BANK DOCUMENTS...","BAG HAVING CASH, LUNCH, BOTTLE, BANK DOCUMENTS...",1,15/12/2021,20:05:00


In [1]:
import re
from difflib import get_close_matches
from pathlib import Path

import pandas as pd


SYNONYMS = {
    "mobile": "smartphone", "cellphone": "smartphone", "cell phone": "smartphone",
    "phone": "smartphone", "school bag": "backpack", "rucksack": "backpack",
    "handbag": "purse", "wallet": "purse", "ear phone": "earbuds",
    "ear phone case": "earbuds case", "grey": "gray",
    "dark coloured": "dark", "dark-colored": "dark",
}

ATTRIBUTE_VALUES = {
    "category": {"backpack", "bag", "purse", "wallet", "smartphone", "laptop", "tablet", "earbuds", "earbuds case", "watch", "jacket", "chain", "keys", "bottle", "umbrella", "book", "id card"},
    "colour": {"black", "white", "red", "blue", "green", "yellow", "brown", "gray", "pink", "purple", "orange", "silver", "gold", "dark"},
    "material": {"metal", "leather", "plastic", "canvas", "jute", "denim", "cotton"},
    "features": {"red zipper", "side pocket", "bottle pocket", "charger", "tiffin", "lunch box", "bank card", "aadhar card", "pan card", "earbuds case"},
}


def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        raise TypeError("description must be a string")
    normalized = re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()
    for source, target in sorted(SYNONYMS.items(), key=lambda pair: -len(pair[0])):
        normalized = re.sub(rf"\b{re.escape(source)}\b", target, normalized)
    return re.sub(r"\s+", " ", normalized).strip()


def _find_values(text: str, vocabulary: set[str]) -> list[str]:
    tokens = set(text.split())
    values = []
    for value in vocabulary:
        if re.search(rf"\b{re.escape(value)}\b", text) or set(value.split()).issubset(tokens):
            values.append(value)
    return sorted(values)


def _correct_common_tokens(text: str) -> str:
    known_tokens = sorted({token for values in ATTRIBUTE_VALUES.values() for value in values for token in value.split()})
    corrected = []
    for token in text.split():
        match = get_close_matches(token, known_tokens, n=1, cutoff=0.86)
        corrected.append(match[0] if match else token)
    return " ".join(corrected)


def extract_attributes(report: dict) -> dict:
    if not isinstance(report, dict):
        raise TypeError("report must be a dictionary")
    combined = normalize_text(f"{report.get('item_name', '')} {report.get('description', '')}")
    corrected = _correct_common_tokens(combined)
    features = _find_values(corrected, ATTRIBUTE_VALUES["features"])
    colour_text = corrected
    for feature in features:
        colour_text = re.sub(rf"\b{re.escape(feature)}\b", " ", colour_text)
    return {
        "category": _find_values(corrected, ATTRIBUTE_VALUES["category"]),
        "brand": report.get("brand"),
        "colour": _find_values(colour_text, ATTRIBUTE_VALUES["colour"]),
        "model": report.get("model"),
        "material": _find_values(corrected, ATTRIBUTE_VALUES["material"]),
        "distinctive_features": features,
        "location": report.get("location"),
        "reported_at": report.get("reported_at"),
        "normalized_text": corrected,
    }


def load_found_reports(csv_path: str | Path) -> list[dict]:
    frame = pd.read_csv(csv_path)
    required = {"item_name", "description", "station_name", "receiving_date", "receiving_time"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {sorted(missing)}")
    reports = []
    for report_id, row in frame.iterrows():
        report = {
            "report_id": f"found-{report_id}", "report_type": "found",
            "item_name": str(row["item_name"]), "description": str(row["description"]),
            "location": str(row["station_name"]),
            "reported_at": f"{row['receiving_date']} {row['receiving_time']}",
        }
        report["attributes"] = extract_attributes(report)
        reports.append(report)
    return reports


found_reports = load_found_reports(Path("delhimetrorail (1).csv"))
semantic_variant = extract_attributes({"description": "dark-colored school bag with a side bottle pocket"})
spelling_variant = extract_attributes({"description": "blak backpak with charger"})
conflicting_variant = extract_attributes({"description": "red backpack"})

assert "backpack" in semantic_variant["category"]
assert "dark" in semantic_variant["colour"]
assert "side pocket" in semantic_variant["distinctive_features"]
assert "backpack" in spelling_variant["category"]
assert "black" in spelling_variant["colour"]
assert "red" in conflicting_variant["colour"]
assert len(found_reports) > 0

print(f"Loaded {len(found_reports):,} found reports")
print("Semantic variant:", semantic_variant)
print("Spelling variant:", spelling_variant)
print("Conflicting colour remains explicit:", conflicting_variant["colour"])


Loaded 13,713 found reports
Semantic variant: {'category': ['backpack', 'bottle'], 'brand': None, 'colour': ['dark'], 'model': None, 'material': [], 'distinctive_features': ['bottle pocket', 'side pocket'], 'location': None, 'reported_at': None, 'normalized_text': 'dark colored backpack with a side bottle pocket'}
Spelling variant: {'category': ['backpack'], 'brand': None, 'colour': ['black'], 'model': None, 'material': [], 'distinctive_features': ['charger'], 'location': None, 'reported_at': None, 'normalized_text': 'black backpack with charger'}
Conflicting colour remains explicit: ['red']


In [3]:
from datetime import timedelta


def _token_similarity(left: str, right: str) -> float:
    left_tokens = set(normalize_text(left).split())
    right_tokens = set(normalize_text(right).split())
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)


def _attribute_relation(lost_values: list[str], found_values: list[str]) -> tuple[float | None, list[str], list[str]]:
    if not lost_values or not found_values:
        return None, [], []
    shared = sorted(set(lost_values) & set(found_values))
    conflicts = sorted(set(lost_values) - set(found_values)) if not shared else []
    return (1.0 if shared else 0.0), shared, conflicts


def _date_relation(lost_at: str | None, found_at: str | None) -> float | None:
    if not lost_at or not found_at:
        return None
    lost_date = pd.to_datetime(lost_at, dayfirst=True, errors="coerce")
    found_date = pd.to_datetime(found_at, dayfirst=True, errors="coerce")
    if pd.isna(lost_date) or pd.isna(found_date):
        return None
    days = abs((lost_date - found_date).days)
    return 1.0 if days <= 3 else 0.5 if days <= 14 else 0.0


def _location_relation(lost_location: str | None, found_location: str | None) -> float | None:
    if not lost_location or not found_location:
        return None
    return 1.0 if _token_similarity(lost_location, found_location) >= 0.5 else 0.0


def _score_candidate(lost_report: dict, found_report: dict) -> dict:
    lost_attributes = lost_report.get("attributes") or extract_attributes(lost_report)
    found_attributes = found_report.get("attributes") or extract_attributes(found_report)
    evidence = []
    shared = []
    conflicts = []
    text_score = _token_similarity(lost_attributes.get("normalized_text", ""), found_attributes.get("normalized_text", ""))
    evidence.append((25, text_score))

    field_weights = {"category": 15, "brand": 10, "colour": 10, "material": 5, "distinctive_features": 20}
    for field, weight in field_weights.items():
        relation, field_shared, field_conflicts = _attribute_relation(lost_attributes.get(field, []), found_attributes.get(field, []))
        if relation is not None:
            evidence.append((weight, relation))
            shared.extend(f"{field}: {value}" for value in field_shared)
            conflicts.extend(f"{field}: {value}" for value in field_conflicts)

    location_score = _location_relation(lost_attributes.get("location"), found_attributes.get("location"))
    if location_score is not None:
        evidence.append((10, location_score))
        if location_score:
            shared.append(f"location: {found_attributes['location']}")
    date_score = _date_relation(lost_attributes.get("reported_at"), found_attributes.get("reported_at"))
    if date_score is not None:
        evidence.append((5, date_score))
        if date_score:
            shared.append("compatible report dates")

    total_weight = sum(weight for weight, _ in evidence)
    raw_score = 100 * sum(weight * value for weight, value in evidence) / total_weight
    score = max(0, min(100, round(raw_score - 15 * len(conflicts))))
    has_anchor = any(value.startswith("category:") or value.startswith("distinctive_features:") for value in shared)
    explanation = []
    if shared:
        explanation.append("Shared " + ", ".join(shared[:4]))
    if conflicts:
        explanation.append("Conflicts " + ", ".join(conflicts[:4]))
    if not explanation:
        explanation.append("Only limited generic description evidence was available")
    return {
        "found_report_id": found_report.get("report_id"),
        "candidate_description": found_report.get("description", ""),
        "score": score,
        "shared_characteristics": shared,
        "conflicting_details": conflicts,
        "explanation": ". ".join(explanation) + ".",
        "requires_human_verification": True,
        "_has_anchor": has_anchor,
    }


def rank_matches(lost_report: dict, found_reports: list[dict], top_k: int = 5, threshold: int = 45) -> dict:
    if not isinstance(lost_report, dict):
        raise TypeError("lost_report must be a dictionary")
    if top_k < 1 or top_k > 20:
        raise ValueError("top_k must be between 1 and 20")
    candidates = [_score_candidate(lost_report, report) for report in found_reports]
    candidates.sort(key=lambda candidate: candidate["score"], reverse=True)
    selected = [candidate for candidate in candidates if candidate["score"] >= threshold][:top_k]
    strong = bool(selected and selected[0]["score"] >= 60 and selected[0]["_has_anchor"] and not selected[0]["conflicting_details"])
    for candidate in selected:
        candidate.pop("_has_anchor")
    return {"status": "potential_matches" if strong else "no_strong_match", "message": None if strong else "No strong match found", "matches": selected}


def _synthetic_found(report_id: str, description: str, location: str, reported_at: str) -> dict:
    report = {"report_id": report_id, "report_type": "found", "description": description, "location": location, "reported_at": reported_at}
    report["attributes"] = extract_attributes(report)
    return report


lost_test_report = {"report_type": "lost", "description": "black backpack with a red zipper", "location": "main library", "reported_at": "08/10/2026"}
synthetic_found = [
    _synthetic_found("found-exact", "Black backpack with red zipper", "main library", "08/10/2026"),
    _synthetic_found("found-semantic", "Dark school bag with red zipper", "main library", "09/10/2026"),
    _synthetic_found("found-conflict", "Blue backpack with side pocket", "main library", "08/10/2026"),
    _synthetic_found("found-unrelated", "Grey jacket", "sports centre", "08/10/2026"),
]
ranking_result = rank_matches(lost_test_report, synthetic_found, top_k=4)
assert ranking_result["status"] == "potential_matches"
assert ranking_result["matches"][0]["found_report_id"] == "found-exact"
assert ranking_result["matches"][1]["found_report_id"] == "found-semantic"
conflict_result = _score_candidate(lost_test_report, synthetic_found[2])
assert any("colour" in detail for detail in conflict_result["conflicting_details"])

incomplete_result = rank_matches({"description": "backpack"}, synthetic_found, top_k=3)
assert incomplete_result["matches"]

print("Ranking status:", ranking_result["status"])
for match in ranking_result["matches"]:
    print(match["found_report_id"], match["score"], match["explanation"])
print("Conflicting candidate score:", conflict_result["score"])


Ranking status: potential_matches
found-exact 95 Shared category: backpack, colour: black, distinctive_features: red zipper, location: main library.
found-semantic 61 Shared category: backpack, distinctive_features: red zipper, location: main library, compatible report dates. Conflicts colour: black.
Conflicting candidate score: 12


In [4]:
def evaluate_ranker(evaluation_cases: list[dict], found_reports: list[dict], k: int = 3) -> dict:
    if k < 1:
        raise ValueError("k must be at least 1")
    precisions = []
    recalls = []
    reciprocal_ranks = []
    case_results = []

    for case in evaluation_cases:
        expected = set(case["relevant_found_ids"])
        result = rank_matches(case["lost_report"], found_reports, top_k=k, threshold=0)
        returned_ids = [match["found_report_id"] for match in result["matches"][:k]]
        returned_set = set(returned_ids)
        hits = len(returned_set & expected)
        precisions.append(hits / k)
        recalls.append(hits / len(expected) if expected else 1.0)
        first_relevant_rank = next((index + 1 for index, report_id in enumerate(returned_ids) if report_id in expected), None)
        reciprocal_ranks.append(1 / first_relevant_rank if first_relevant_rank else 0.0)
        case_results.append({"case": case["name"], "returned_ids": returned_ids, "hits": hits})

    return {
        "precision_at_k": sum(precisions) / len(precisions),
        "recall_at_k": sum(recalls) / len(recalls),
        "mean_reciprocal_rank": sum(reciprocal_ranks) / len(reciprocal_ranks),
        "k": k,
        "cases": case_results,
    }


evaluation_cases = [
    {
        "name": "exact description",
        "lost_report": {"description": "black backpack with red zipper", "location": "main library", "reported_at": "08/10/2026"},
        "relevant_found_ids": ["found-exact"],
    },
    {
        "name": "semantic wording",
        "lost_report": {"description": "dark school bag with a red zip", "location": "main library", "reported_at": "09/10/2026"},
        "relevant_found_ids": ["found-exact", "found-semantic"],
    },
    {
        "name": "conflicting colour",
        "lost_report": {"description": "blue backpack with side pocket", "location": "main library", "reported_at": "08/10/2026"},
        "relevant_found_ids": ["found-conflict"],
    },
    {
        "name": "incomplete report",
        "lost_report": {"description": "backpack"},
        "relevant_found_ids": ["found-exact", "found-semantic", "found-conflict"],
    },
]

evaluation_result = evaluate_ranker(evaluation_cases, synthetic_found, k=3)
assert 0 <= evaluation_result["precision_at_k"] <= 1
assert 0 <= evaluation_result["recall_at_k"] <= 1
assert 0 <= evaluation_result["mean_reciprocal_rank"] <= 1
assert evaluation_result["cases"][0]["hits"] == 1

no_match_result = rank_matches({"description": "green laptop with silver case"}, synthetic_found, top_k=3)
assert no_match_result["status"] == "no_strong_match"

print(f"Precision@3: {evaluation_result['precision_at_k']:.2f}")
print(f"Recall@3: {evaluation_result['recall_at_k']:.2f}")
print(f"Mean reciprocal rank: {evaluation_result['mean_reciprocal_rank']:.2f}")
for case in evaluation_result["cases"]:
    print(case["case"], "->", case["returned_ids"], "hits:", case["hits"])
print("No-match status:", no_match_result["status"])


Precision@3: 0.58
Recall@3: 1.00
Mean reciprocal rank: 1.00
exact description -> ['found-exact', 'found-semantic', 'found-conflict'] hits: 1
semantic wording -> ['found-semantic', 'found-exact', 'found-conflict'] hits: 2
conflicting colour -> ['found-conflict', 'found-exact', 'found-semantic'] hits: 1
incomplete report -> ['found-exact', 'found-semantic', 'found-conflict'] hits: 3
No-match status: no_strong_match


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def embedding_rank_matches(lost_report: dict, found_reports: list[dict], top_k: int = 5, threshold: int = 45) -> dict:
    if not isinstance(lost_report, dict):
        raise TypeError("lost_report must be a dictionary")
    if not found_reports:
        return {"status": "no_strong_match", "message": "No strong match found", "matches": []}

    lost_attributes = lost_report.get("attributes") or extract_attributes(lost_report)
    found_attributes = [report.get("attributes") or extract_attributes(report) for report in found_reports]
    lost_text = lost_attributes.get("normalized_text", "")
    found_texts = [attributes.get("normalized_text", "") for attributes in found_attributes]
    vectors = embedding_model.encode([lost_text, *found_texts], normalize_embeddings=True)
    semantic_scores = np.dot(vectors[1:], vectors[0])

    candidates = []
    for report, semantic_score in zip(found_reports, semantic_scores):
        baseline = _score_candidate(lost_report, report)
        combined_score = round(40 * max(0.0, float(semantic_score)) + 60 * baseline["score"] / 100)
        baseline["score"] = max(0, min(100, combined_score))
        baseline["semantic_similarity"] = round(float(semantic_score), 3)
        candidates.append(baseline)

    candidates.sort(key=lambda candidate: candidate["score"], reverse=True)
    selected = [candidate for candidate in candidates if candidate["score"] >= threshold][:top_k]
    strong = bool(selected and selected[0]["score"] >= 60 and selected[0]["_has_anchor"] and not selected[0]["conflicting_details"])
    for candidate in selected:
        candidate.pop("_has_anchor")
    return {
        "status": "potential_matches" if strong else "no_strong_match",
        "message": None if strong else "No strong match found",
        "matches": selected,
    }


embedding_result = embedding_rank_matches(lost_test_report, synthetic_found, top_k=4)
assert embedding_result["matches"]
assert embedding_result["matches"][0]["found_report_id"] == "found-exact"
assert all(match["requires_human_verification"] for match in embedding_result["matches"])

print("Embedding model: all-MiniLM-L6-v2")
for match in embedding_result["matches"]:
    print(match["found_report_id"], match["score"], "semantic similarity:", match["semantic_similarity"])


c:\Users\louis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1438.79it/s]


Embedding model: all-MiniLM-L6-v2
found-exact 97 semantic similarity: 0.994
found-semantic 74 semantic similarity: 0.928


In [6]:
def evaluate_matcher(matcher, evaluation_cases: list[dict], candidates: list[dict], k: int = 3) -> dict:
    precisions = []
    recalls = []
    reciprocal_ranks = []
    for case in evaluation_cases:
        result = matcher(case["lost_report"], candidates, top_k=k, threshold=0)
        expected = set(case["relevant_found_ids"])
        returned_ids = [match["found_report_id"] for match in result["matches"][:k]]
        hits = len(set(returned_ids) & expected)
        precisions.append(hits / k)
        recalls.append(hits / len(expected) if expected else 1.0)
        first_rank = next((index + 1 for index, report_id in enumerate(returned_ids) if report_id in expected), None)
        reciprocal_ranks.append(1 / first_rank if first_rank else 0.0)
    return {
        "precision_at_k": sum(precisions) / len(precisions),
        "recall_at_k": sum(recalls) / len(recalls),
        "mean_reciprocal_rank": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


baseline_comparison = evaluate_matcher(rank_matches, evaluation_cases, synthetic_found, k=3)
embedding_comparison = evaluate_matcher(embedding_rank_matches, evaluation_cases, synthetic_found, k=3)
assert set(baseline_comparison) == set(embedding_comparison)

print("Baseline metrics:", baseline_comparison)
print("Embedding hybrid metrics:", embedding_comparison)


Baseline metrics: {'precision_at_k': 0.5833333333333333, 'recall_at_k': 1.0, 'mean_reciprocal_rank': 1.0}
Embedding hybrid metrics: {'precision_at_k': 0.5833333333333333, 'recall_at_k': 1.0, 'mean_reciprocal_rank': 1.0}
